Task 1: Update the Character class
When we created the Character class in the lecture we first included an attack()
method, and afterwards indicated that we wanted hit points to be a protected feature
_hp . This lead to the Character.attack() method changing the target's protected
attribute. Modifying another object's private attribute is bad form! Your first task is to fix
this.
- Implement a take_damage() method to be used when the Character object is injured. This method should only affect the characters own HP.
- Implement a heal() method to be used when the Character heals (in other words increase their hit points)
- Change the existing attack() method so that calls the target's take_damage() method rather than directly modifying the target's _hp .
- Add an is_alive property to the Character class. This should be read-only, and return True if _hp and False otherwise.

In [ ]:
# Object-oriented game simulator
# Turid Torheim, NMBU, 2026

import math as m

class Character:
    """Represents a character in the RPG"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, healing: int = 5, weapon=None) -> None:
        """
        Initialise the Character
        Args:
        name (str): The name of the character.
        hp_max (int): Max hit points (default = 100)
        damage (int): Damage delt when character attacks (default = 10)
        """
        self.name = name
        self._hp = hp_max # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.healing = healing
        self.weapon = weapon

    def take_damage(self, damage: int = 10) -> None:
        if self._hp <= 0:
            print(f"The {self.name} is dead")
        else:
            self._hp -= damage
            self._hp = m.max(self._hp, 0)

    def heal(self, healing: int = 5) -> None:
        self._hp += healing
        self._hp = m.min(self._hp, self.max_hp)

    def attack(self, target) -> None:
        """Attack and deal damage to target character"""
        # target._hp -= self.damage
        # target._hp = max(target._hp, 0) # Target hp cannot end up lower than 0
        # Du "står på" target som gjør skade mot self-karakteren (ikke deg selv altså)
        # Du kaller metoden på det andre objektet/karakteren
        # target.take_damage(self.damage)

        # Mitt forslag
        if self.weapon is not None:
            damage_from_weapon = self.weapon.roll_damage()
            target.take_damage(damage_from_weapon)
        else:
            target.take_damage(self.damage)


    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe
        edits."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        if self._hp > 0:
            return True
        return False



In [270]:
Warrior = Character("Warrior")
Mage = Character("Mage")

print(Warrior.__repr__())
print(repr(Warrior))

Warrior.attack(Mage)
print(Mage.hp)

<Character name=Warrior hp=100 damage=10>
<Character name=Warrior hp=100 damage=10>
90


Task 2: Introducing randomness with a Dice class
- Currently, characters deal fixed damage. To introduce variable damage, create a separate Dice class.

- Create a Dice class that can create any n-sided Dice (n>=2), where the default number of sides is 6.

- Add a roll() method to roll the dice, using random.randint(1, self.sides) .

- Add a __repr__ method that returns string representations like Dice(sides=6) .

- Implement a __call__ method that allow you to roll the dice created by d20 = Dice(sides=20) by calling d20.roll() .

    - See the Python documentation of __call__ or this example

In [259]:
import random


class Dice:

    def __init__(self, sides: int = 6):
        if sides < 2:
            raise ValueError("The die has to have at least two sides")
        self.sides = sides

    def roll(self):
        self.eyes = random.randint(1, self.sides)
        return self.eyes

    def __repr__(self):
        return f"Dice(sides={self.sides})"

    def __call__(self):
        # self.eyes = self.roll()
        return self.roll()

In [261]:
# dice_test = Dice(1)

dice = Dice(6)
dice.roll()

print(dice())
print(dice)

d20 = Dice(sides=20)
print(d20)
print(d20())

3
Dice(sides=6)
Dice(sides=20)
16


Task 3: Create items
Create a Weapon class in items.py that utilizes your Dice class for damage calculations. We call the module items.py rather than weapons.py so that it can be expanded to include other items your characters can hold.

- Define the class Weapon with the attributes name and damage_dice: Dice .

- Include a roll_damage() method that calls self.damage_dice.roll()

- Update Character.__init__ to accept an optional weapon.

- If a character has an equipped weapon, attack() should roll damage from the weapon instead of using default base damage.

In [244]:
# Add to a separate file called items.py
# TODO: import the file with the other class to this one

class Weapon:

    def __init__(self, name: str, damage_dice: Dice):
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self):
        return self.damage_dice.roll()

    # def __call__(self):
        # return self.roll_damage()

In [254]:
sword = Weapon("sword", d20)

# roll = sword.roll_damage()

# print(roll)

Warrior = Character("Warrior", weapon=sword)
Mage = Character("Mage")


Warrior.attack(Mage)
print(Mage.hp)

print(Mage)

91
Character: Mage with 91 HP and 10 damage


Task 4: Character specialization
Extend the character module to include two specialised subclasses of Character .

1. Warrior subclass (defensive specialist)
   - New attribute: armor: int , with a default value of 5
   - Override the Character.take_damage() method by reducing the incoming damage by armor before applying it.
2. Mage subclass (resource-based attacker)
   - New attributes: mana: int and mana_max: int , both with default value 30
   - Magic attacks consume mana, so the mana available determines whether the mage can use magic.
   - Override the Character.attack() method:
      - If the mage has at least 10 mana left, spend 10 mana to deal a magical spellattack dealing double damage. This magic attack does not use any weapons, even if the character has one equipped.
      - If the mage has less than 10 mana left, fall back to the base attack, and restore 5 mana. This base attack should use whichever weapon the character has equipped.


In [ ]:
class Warrior(Character):

    def __init__(self, name, hp_max=100, damage=10, healing=5, weapon=None, armor=5):
        super().__init__(name, hp_max, damage, healing, weapon)
        self.armor = armor

    def take_damage(self, damage: int = 10) -> None:
        real_damage = m.max(damage - self.armor, 0)
        super().take_damage(real_damage)


class Mage(Character):

    def __init__(self, name, hp_max = 100, damage = 10, healing = 5, weapon=None, mana_max=30, mana=30):
        super().__init__(name, hp_max, damage, healing, weapon)
        self.mana = mana
        self.mana_max = mana_max

    def attack(self, target) -> None:
        if self.mana >= 10:
            target.take_damage(self.damage * 2)
            self.mana = m.max(self.mana - 10, 0)
        else:
            super().attack(target)
            self.mana = m.min(self.mana + 5, self.mana_max)


Warrior = Character("Warrior", weapon=sword)
Mage = Character("Mage")

Mage.attack(Warrior)


Task 5: Running battles
Create a Battle class to manage turn-based combat between two Character objects.
- Initialize Battle with hero: Character and enemy: Character .
- Implement a play_round() method where hero attacks enemy , and if enemy is still alive, enemy attacks hero .
- Implement a simulate_battle() method with a while loop that runs rounds until one character is no longer alive.